<a href="https://colab.research.google.com/github/Abhirup-kar/Stellar-Class-Predictor-kaggle-competition-/blob/main/stellar_prediction_pytorch_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here We use pytorch (nural model)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install opendatasets --quiet

In [3]:
import pandas as pd

In [4]:
import opendatasets as od
od.download('https://www.kaggle.com/competitions/playground-series-s6e6')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: abhirupkar039
Your Kaggle Key: ··········


100%|██████████| 58.6M/58.6M [00:01<00:00, 30.9MB/s]



Extracting archive ./playground-series-s6e6/playground-series-s6e6.zip to ./playground-series-s6e6


In [5]:
sub_df = pd.read_csv('/content/playground-series-s6e6/sample_submission.csv')

In [6]:
import joblib
stellar_data = joblib.load('/content/drive/MyDrive/stellar_dataset.joblib')

In [7]:
train_inputs = stellar_data['train_inputs']
train_targets = stellar_data['train_targets']
val_inputs = stellar_data['val_inputs']
val_targets = stellar_data['val_targets']
test_inputs = stellar_data['test_inputs']
Lencoder = joblib.load('/content/drive/MyDrive/Lencoder.joblib')

In [ ]:
import torch

X_train_tensor = torch.tensor(train_inputs.values, dtype=torch.float32)
y_train_tensor = torch.tensor(train_targets, dtype=torch.long)

X_val_tensor = torch.tensor(val_inputs.values, dtype=torch.float32)
y_val_tensor = torch.tensor(val_targets, dtype=torch.long)

X_test_tensor = torch.tensor(test_inputs.values, dtype=torch.float32)
# Note: test_targets are not available in 'stellar_data' for accuracy calculation on the test set.

In [ ]:
import torch.nn as nn

class StellarNet(nn.Module):
    def __init__(self, input_size):
        super(StellarNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)

model = StellarNet(train_inputs.shape[1])

Loss Function and Optimizer

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

**Training Loop**

In [ ]:
epochs = 75

for epoch in range(epochs):

    outputs = model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [10/75], Loss: 0.9937
Epoch [20/75], Loss: 0.9475
Epoch [30/75], Loss: 0.8772
Epoch [40/75], Loss: 0.7778
Epoch [50/75], Loss: 0.6702
Epoch [60/75], Loss: 0.5996
Epoch [70/75], Loss: 0.5696


Prediction

In [ ]:
with torch.no_grad():
    # Make predictions on the actual test inputs
    test_outputs = model(X_test_tensor)
    _, predicted = torch.max(test_outputs, 1)
# The 'predicted' variable now holds the predictions for your 'test_inputs'.

The `predicted` tensor contains the class indices from your model's predictions on the `test_inputs`. To see the actual class labels, we can use the `Lencoder` to inverse transform these indices.

In [ ]:
# Decode the predicted class indices into original class labels
predicted_labels = Lencoder.inverse_transform(predicted.numpy())

print("First 10 predicted labels for test_inputs:")
print(predicted_labels[:10])

# You can also get a count of each predicted class
from collections import Counter
print("\nDistribution of predicted labels:")
print(Counter(predicted_labels))

First 10 predicted labels for test_inputs:
['GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY' 'GALAXY'
 'GALAXY' 'GALAXY']

Distribution of predicted labels:
Counter({'GALAXY': 177864, 'QSO': 69571})


Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

# To assess the model's performance, we'll calculate accuracy on the validation set.
# Since 'test_targets' are not available, we cannot compute accuracy for the 'test_inputs'.
with torch.no_grad():
    val_outputs = model(X_val_tensor)
    _, val_predicted = torch.max(val_outputs, 1)

accuracy = accuracy_score(
    y_val_tensor.numpy(),
    val_predicted.numpy()
)

print(f"Validation Accuracy: {accuracy:.4f}")


Validation Accuracy: 0.7830


In [ ]:
predicted_labels

array(['GALAXY', 'GALAXY', 'GALAXY', ..., 'GALAXY', 'GALAXY', 'GALAXY'],
      dtype=object)

In [ ]:
sub_df['class'] = predicted_labels

In [ ]:
sub_df.to_csv('submission_NLP.csv', index=False)

Xgboost model tuning further

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42
)

param_grid = {
    "n_estimators": [300, 500, 700, 1000],
    "max_depth": [4, 6, 8, 10, 12],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7],
    "gamma": [0, 0.1, 0.2, 0.3]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,
    cv=3,
    scoring="accuracy",
    verbose=2,
    random_state=42
)

random_search.fit(train_inputs,train_targets)
print(random_search.best_params_)
print(random_search.best_score_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [13:52:29] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   6.8s
[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   7.7s
[CV] END colsample_bytree=1.0, gamma=0, learning_rate=0.03, max_depth=6, min_child_weight=7, n_estimators=300, subsample=1.0; total time=   3.2s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.6s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.8s
[CV] END colsample_bytree=0.7, gamma=0, learning_rate=0.05, max_depth=10, min_child_weight=3, n_estimators=1000, subsample=0.7; total time=  19.6s
[CV] END colsample_bytree=0.8, gamma=0, learning_rate=0.01, max_depth=12, min_child_weight=1, n_estimators=1000, subsample=0

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
accuracy_score(val_targets,best_model.predict(val_inputs))

0.9686065644756213

In [ ]:
test_preds = best_model.predict(test_inputs)

In [ ]:
test_preds = Lencoder.inverse_transform(test_preds)

In [ ]:
sub_df['class'] = test_preds10
sub_df.to_csv("random_forest_tune_sub.csv",index = False)

In [ ]:
sub_df['class'] = test_preds

In [ ]:
sub_df.to_csv("xgboost_fur_tune.csv",index = False)

Further Tuning LightGBM Model

In [ ]:
!pip install lightgbm --quiet

In [ ]:
from lightgbm import LGBMClassifier

lgb_gpu = LGBMClassifier(
    objective='multiclass',
    num_class=3,              # Change if your number of classes differs
    boosting_type='gbdt',

    device='gpu',

    n_estimators=500,
    learning_rate=0.05,

    num_leaves=127,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    verbose=2
)

param_dist = {
    "n_estimators": [300, 500, 700,1000],
    "learning_rate": [ 0.03, 0.05, 0.1,0.3],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 10, 20],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_samples": [10, 20, 30]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(
    estimator=lgb_gpu,
    param_distributions=param_dist,
    n_iter=15,
    cv=3,
    scoring="accuracy",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(train_inputs, train_targets)
print(search.best_params_)
print(search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2052
[LightGBM] [Info] Number of data points in the train set: 461877, number of used features: 14
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 10 dense feature groups (5.29 MB) transferred to GPU in 0.008525 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score -0.425580
[LightGBM] [Info] Start training from score -1.593068
[LightGBM] [Info] Start training from score -1.942754
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 16
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 13
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth = 16
[LightGBM] [Debug] Trained a tree with leaves = 255 and depth =

In [ ]:
best_model = search.best_estimator_

val_preds = best_model.predict(val_inputs)

In [ ]:
accuracy_score(val_targets,val_preds)

0.9693946479605092

In [ ]:
test_preds_lgbm = best_model.predict(test_inputs)

In [ ]:
test_preds_lgbm = Lencoder.inverse_transform(test_preds_lgbm)

In [ ]:
sub_df['class'] = test_preds_lgbm

In [ ]:
sub_df.to_csv("lgbm_tune.csv",index = False)

Catboost Model

In [ ]:
!pip install catboost  optuna --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.3 MB/s eta 0:00:00


In [ ]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 500, 3000),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 20
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0, 10
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 10
        ),
        "border_count": trial.suggest_int(
            "border_count", 32, 255
        ),
        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf", 1, 50
        ),

        # Multiclass settings
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "random_seed": 42,
        "verbose": 0
    }

    model = CatBoostClassifier(
        **params,
        task_type="GPU",

    )

    model.fit(
        train_inputs,
        train_targets,
        eval_set=(val_inputs, val_targets),
        early_stopping_rounds=100,
        verbose=False
    )

    preds = model.predict(val_inputs)

    return accuracy_score(val_targets, preds)


study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=75,
    show_progress_bar=True
)

print("Best Score:", study.best_value)
print("Best Parameters:")
print(study.best_params)

[I 2026-06-24 12:52:37,323] A new study created in memory with name: no-name-a01cb1ec-b326-4a07-9167-5ab927d4a15b


  0%|          | 0/75 [00:00<?, ?it/s]

[I 2026-06-24 12:53:40,147] Trial 0 finished with value: 0.963661557114402 and parameters: {'iterations': 1736, 'learning_rate': 0.08884618698552975, 'depth': 10, 'l2_leaf_reg': 2.6890009378803885, 'random_strength': 9.324209557408103, 'bagging_temperature': 6.193485590264909, 'border_count': 125, 'min_data_in_leaf': 12}. Best is trial 0 with value: 0.963661557114402.
[I 2026-06-24 12:53:57,935] Trial 1 finished with value: 0.9571663635576341 and parameters: {'iterations': 1852, 'learning_rate': 0.04351172248240744, 'depth': 5, 'l2_leaf_reg': 17.537147367087247, 'random_strength': 3.7128324856647, 'bagging_temperature': 5.541195504021634, 'border_count': 145, 'min_data_in_leaf': 3}. Best is trial 0 with value: 0.963661557114402.
[I 2026-06-24 12:54:08,086] Trial 2 finished with value: 0.9391443665021217 and parameters: {'iterations': 1014, 'learning_rate': 0.013051504059545884, 'depth': 4, 'l2_leaf_reg': 15.70571782599611, 'random_strength': 2.2151394756076934, 'bagging_temperature': 9

In [ ]:
import pandas as pd

X_final = pd.concat([train_inputs, val_inputs])
y_final = pd.concat([pd.Series(train_targets), pd.Series(val_targets)])

best_params = study.best_params

model = CatBoostClassifier(
    **best_params,
    task_type="GPU",
    loss_function="MultiClass",
    eval_metric="MultiClass",
    random_seed=42
)

model.fit(X_final, y_final)

0:	learn: 0.8594897	total: 17ms	remaining: 32s
1:	learn: 0.7059958	total: 33ms	remaining: 31.1s
2:	learn: 0.5991517	total: 48.8ms	remaining: 30.6s
3:	learn: 0.5190338	total: 64.3ms	remaining: 30.2s
4:	learn: 0.4566756	total: 83.9ms	remaining: 31.5s
5:	learn: 0.4075329	total: 116ms	remaining: 36.4s
6:	learn: 0.3688771	total: 136ms	remaining: 36.4s
7:	learn: 0.3367324	total: 152ms	remaining: 35.6s
8:	learn: 0.3096919	total: 167ms	remaining: 34.8s
9:	learn: 0.2881785	total: 182ms	remaining: 34.2s
10:	learn: 0.2705681	total: 196ms	remaining: 33.4s
11:	learn: 0.2546755	total: 212ms	remaining: 33.1s
12:	learn: 0.2409337	total: 230ms	remaining: 33.1s
13:	learn: 0.2292775	total: 247ms	remaining: 33s
14:	learn: 0.2198902	total: 262ms	remaining: 32.6s
15:	learn: 0.2120005	total: 275ms	remaining: 32.2s
16:	learn: 0.2037710	total: 286ms	remaining: 31.4s
17:	learn: 0.1983157	total: 298ms	remaining: 30.9s
18:	learn: 0.1922154	total: 310ms	remaining: 30.5s
19:	learn: 0.1860613	total: 322ms	remaining:

CatBoostClassifier(bagging_temperature=0.4318038690515593, border_count=218, depth=6, eval_metric='MultiClass', iterations=1885, l2_leaf_reg=12.266998420972184, learning_rate=0.16509310986170173, loss_function='MultiClass', min_data_in_leaf=42, random_seed=42, random_strength=2.3984392143602338, task_type='GPU')

In [ ]:
test_predictions = model.predict(test_inputs)
test_probabilities = model.predict_proba(test_inputs)

In [ ]:
test_preds_catboost = Lencoder.inverse_transform(test_predictions)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
test_preds_catboost

array(['GALAXY', 'GALAXY', 'GALAXY', ..., 'GALAXY', 'QSO', 'GALAXY'],
      dtype=object)

In [ ]:
sub_df['class'] = test_preds_catboost

In [ ]:
sub_df.to_csv("catboost_tune.csv",index = False)

LGBM model

In [9]:
!pip install optuna lightgbm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 17.5 MB/s eta 0:00:00


In [ ]:
import optuna
import lightgbm as lgb
from sklearn.metrics import accuracy_score

NUM_CLASSES = 3

def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": NUM_CLASSES,
        "metric": "multi_logloss",

        # GPU
        "device": "gpu",

        # Core parameters
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.2, log=True
        ),
        "n_estimators": trial.suggest_int(
            "n_estimators", 300, 3000
        ),
        "num_leaves": trial.suggest_int(
            "num_leaves", 31, 255
        ),
        "max_depth": trial.suggest_int(
            "max_depth", -1, 16
        ),

        # Regularization
        "min_child_samples": trial.suggest_int(
            "min_child_samples", 5, 100
        ),
        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1e-3, 10, log=True
        ),
        "lambda_l1": trial.suggest_float(
            "lambda_l1", 1e-8, 10, log=True
        ),
        "lambda_l2": trial.suggest_float(
            "lambda_l2", 1e-8, 10, log=True
        ),

        # Sampling
        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.6, 1.0
        ),
        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.6, 1.0
        ),
        "bagging_freq": trial.suggest_int(
            "bagging_freq", 1, 10
        ),

        "verbosity": -1,
        "random_state": 42
    }

    model = lgb.LGBMClassifier(**params)

    model.fit(
        train_inputs,
        train_targets,
        eval_set=[(val_inputs, val_targets)],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(100, verbose=False)
        ]
    )

    preds = model.predict(val_inputs)

    return accuracy_score(val_targets, preds)


study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=75,
    show_progress_bar=True
)

print(study.best_value)
print(study.best_params)

[I 2026-06-27 08:55:54,701] A new study created in memory with name: no-name-f71efe8e-2cae-498d-992b-9474cb2e563d


  0%|          | 0/75 [00:00<?, ?it/s]

[I 2026-06-27 09:01:29,569] Trial 0 finished with value: 0.9681215900233827 and parameters: {'learning_rate': 0.07131132052011642, 'n_estimators': 2111, 'num_leaves': 169, 'max_depth': 4, 'min_child_samples': 90, 'min_child_weight': 0.37486516588916813, 'lambda_l1': 1.63558201025706e-08, 'lambda_l2': 0.025608286412263474, 'feature_fraction': 0.6002641739288936, 'bagging_fraction': 0.8124894156678025, 'bagging_freq': 10}. Best is trial 0 with value: 0.9681215900233827.
[I 2026-06-27 09:09:34,678] Trial 1 finished with value: 0.9687711093790595 and parameters: {'learning_rate': 0.010156575347142035, 'n_estimators': 1627, 'num_leaves': 91, 'max_depth': 10, 'min_child_samples': 11, 'min_child_weight': 7.890057373093252, 'lambda_l1': 0.1684028931144182, 'lambda_l2': 0.0017235302076093952, 'feature_fraction': 0.692501994498887, 'bagging_fraction': 0.6201332454769456, 'bagging_freq': 10}. Best is trial 1 with value: 0.9687711093790595.
[I 2026-06-27 09:21:18,595] Trial 2 finished with value: 

In [24]:
import lightgbm as lgb
best_params = {'learning_rate': 0.05234741287658477, 'n_estimators': 2540,
               'num_leaves': 248, 'max_depth': 17, 'min_child_samples': 52,
               'min_child_weight': 9.20737904798539, 'lambda_l1': 0.003380453558372737,
               'lambda_l2': 6.797339657946614, 'feature_fraction': 0.9699498719757521,
               'bagging_fraction': 0.9842049362377556, 'bagging_freq': 1}

model = lgb.LGBMClassifier(
    **best_params,
    objective="multiclass",
    num_class=3,
    device="gpu",
    random_state=42
)

model.fit(
    train_inputs,
    train_targets
)

predictions_lgbm = model.predict(test_inputs)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

In [25]:
from sklearn.metrics import accuracy_score
train_preds11 = model.predict(train_inputs)
accuracy_score(train_targets,train_preds11)

[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9699498719757521, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9699498719757521
[LightGBM] [Warning] lambda_l2 is set=6.797339657946614, reg_lambda=0.0 will be ignored. Current value: lambda_l2=6.797339657946614
[LightGBM] [Warning] lambda_l1 is set=0.003380453558372737, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.003380453558372737
[LightGBM] [Warning] bagging_fraction is set=0.9842049362377556, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9842049362377556


0.9999761841356032

In [26]:
predictions_lgbm = Lencoder.inverse_transform(predictions_lgbm)

In [27]:
sub_df['class'] = predictions_lgbm

In [28]:
sub_df.to_csv("lgbm_fur_tune3.csv",index=False)

Random Forest further models

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 200, 1500, step=100
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 5, 50
        ),

        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 10
        ),

        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None]
        ),

        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            [True, False]
        ),

        "criterion": trial.suggest_categorical(
            "criterion",
            ["gini", "entropy", "log_loss"]
        ),

        "class_weight": trial.suggest_categorical(
            "class_weight",
            [None, "balanced"]
        ),

        "n_jobs": -1,
        "random_state": 42
    }

    model = RandomForestClassifier(
        **params
      )

    model.fit(train_inputs, train_targets)

    preds = model.predict(val_inputs)

    return accuracy_score(val_targets, preds)


study = optuna.create_study(direction="maximize")

study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

print("Best score:", study.best_value)
print("Best params:")
print(study.best_params)

[I 2026-06-24 13:36:56,331] A new study created in memory with name: no-name-3ac131cb-9bc0-4c68-964a-9f96d855b2d6


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-24 13:36:56,348] Trial 0 failed with parameters: {'n_estimators': 1200, 'max_depth': 32, 'min_samples_split': 6, 'min_samples_leaf': 8, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None} because of the following error: TypeError("RandomForestClassifier.__init__() got an unexpected keyword argument 'task_type'").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_788/2419093614.py", line 48, in objective
    model = RandomForestClassifier(
            ^^^^^^^^^^^^^^^^^^^^^^^
TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'task_type'
[W 2026-06-24 13:36:56,350] Trial 0 failed with value None.


TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'task_type'

In [ ]:
import pandas as pd

X_final_rf = pd.concat([train_inputs, val_inputs])
y_final_rf = pd.concat([pd.Series(train_targets), pd.Series(val_targets)])

best_rf = RandomForestClassifier(
    **study.best_params,
    n_jobs=-1,
    random_state=42
)

best_rf.fit(X_final_rf, y_final_rf)

test_predictions_rf = best_rf.predict(test_inputs)

In [ ]:
test_predictions_rf = Lencoder.inverse_transform(test_predictions_rf)

In [ ]:
sub_df['class'] = test_predictions_rf

In [ ]:
sub_df.to_csv("random_forest_tune.csv",index = False)

TODO

Fined Tuning Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

def tune_random_forest(
    X_train,
    y_train,
    cv=3,
    n_iter=15,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1
):
    """
    Tune RandomForest using RandomizedSearchCV.

    Returns:
        best_model
        best_params
        best_score
    """

    rf = RandomForestClassifier(
        random_state=random_state,
        n_jobs=n_jobs
    )

    param_dist = {
        'n_estimators': [200,500,800],
        'max_depth': [10, 15],
        'min_samples_split': [2, 8,15],
        'min_samples_leaf': [3,8],
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True, False]
    }

    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        random_state=random_state,
        n_jobs=n_jobs,
        verbose=2
    )

    search.fit(X_train, y_train)

    return (
        search.best_estimator_,
        search.best_params_,
        search.best_score_
    )

In [ ]:
best_rf, best_params, best_score = tune_random_forest(
    train_inputs,
    train_targets,
    cv=3,
    n_iter=5
)

print("Best Parameters:")
print(best_params)

print("\nBest CV Score:")
print(best_score)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


KeyboardInterrupt: 

In [ ]:
model10_random_forest_tune = best_rf

In [ ]:
val_preds10 = model10_random_forest_tune.predict(val_inputs)
accuracy_score(val_targets,val_preds10)

In [ ]:
test_preds10 = model10_random_forest_tune.predict(test_inputs)

In [ ]:
test_preds10 = Lencoder.inverse_transform(test_preds10)